# Demo Trading Message Data and Limit Order Book

## Understand the Data

The description of the two data files can be found in *MessageandOrderREADME.txt*. Both files have no headers. 

These files provide the trading messages and limit order book data for the stock of Amazon.com, Inc. (Ticker AMZN), during the trading hours (9:30AM to 4PM, US Eastern Time) on one trading day, June 21, 2016.

In [1]:
import pandas as pd
df_orderbook = pd.read_csv('classdata/orderbook.csv',header=None)
df_message = pd.read_csv('classdata/message.csv',header=None)
df_message.columns=["Time","Type","OrderID","Size","Price","Direction"]

In [2]:
df_orderbook.shape

(400391, 40)

In [3]:
df_orderbook.head()

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
0,5859400,200,5853300,18,5859800,200,5853000,150,5861000,200,...,5845300,300,5876500,1160,5843800,200,5879000,500,5842700,300
1,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,...,5846500,300,5876500,1160,5845300,300,5879000,500,5843800,200
2,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,...,5849300,300,5876500,1160,5846500,300,5879000,500,5845300,300
3,5859100,18,5853300,18,5859400,200,5853200,18,5859800,200,...,5849300,300,5873900,100,5846500,300,5876500,1160,5845300,300
4,5859100,18,5853300,18,5859200,18,5853200,18,5859400,200,...,5849300,300,5871000,10,5846500,300,5873900,100,5845300,300


The columns of *orderbook.csv*:

| Column | Meaning | Also called |
| --- | :--- | :--- |
| 0 | Level 1 Ask Price | Best Ask |
| 1 | Level 1 Ask Volume | Best Ask Volume |
| 2 | Level 1 Bid Price | Best Bid |
| 3 | Level 1 Bid Volume | Best Bid Volume |
| 4 | Level 2 Ask Price | 2nd Best Ask |
| 5 | Level 2 Ask Volume | 2nd Best Ask Volume |
| 6 | Level 2 Bid Price | 2nd Best Bid |
| 7 | Level 2 Bid Volume | 2nd Best Bid Volume |
| ... | ... | ... |
| ... | ... | ... |
| 36 | Level 10 Ask Price | 10th Best Ask |
| 37 | Level 10 Ask Volume | 10th Best Ask Volume |
| 38 | Level 10 Bid Price | 10th Best Bid |
| 39 | Level 10 Bid Volume | 10th Best Bid Volume |



In [4]:
df_message.shape

(400391, 6)

In [5]:
df_message.head()

,Time,Type,OrderID,Size,Price,Direction
0,34200.004241,1,16113575,18,5853300,1
1,34200.004261,1,16113584,18,5853200,1
2,34200.004447,1,16113594,18,5853100,1
3,34200.025552,1,16120456,18,5859100,-1
4,34200.025580,1,16120480,18,5859200,-1


The columns of *message.csv*:

    1.) Time: Number of seconds from 00:00am. For example, 34200 means 9:30am
    2.) Type: See below 
    3.) OrderID: A unique ID for each order (not for each message)
    4.) Size: Number of shares
    5.) Price: Dollar price times 10000
    6.) Direction: -1 means Sell; 1 means Buy 

The type of message:

                1: Submission of a new limit order
				2: Cancellation (Partial deletion of a limit order)
				3: Deletion (Total deletion of a limit order)
				4: Execution of a visible limit order			   	 
				5: Execution of a hidden limit order

Row **i** in **df_message** corresponds to row **i** of **df_orderbook**.

## Summary Statistics using Boolean Condition

In [6]:
#The frequency of each type of messages
print(df_message[df_message.Type==1].shape[0])
print(df_message[df_message.Type==2].shape[0])
print(df_message[df_message.Type==3].shape[0])
print(df_message[df_message.Type==4].shape[0])
print(df_message[df_message.Type==5].shape[0])

191015
3260
171126
23658
11332


In [7]:
#The average price of each type of messages
print(df_message[df_message.Type==1].Price.mean())
print(df_message[df_message.Type==2].Price.mean())
print(df_message[df_message.Type==3].Price.mean())
print(df_message[df_message.Type==4].Price.mean())
print(df_message[df_message.Type==5].Price.mean())

5832010.823233778
5828021.226993865
5832533.5951287355
5824440.641643419
5830159.971761384


In [8]:
#The number of unique orders involved in each type of message
print(df_message[df_message.Type==1].OrderID.nunique())
print(df_message[df_message.Type==2].OrderID.nunique())
print(df_message[df_message.Type==3].OrderID.nunique())
print(df_message[df_message.Type==4].OrderID.nunique())
print(df_message[df_message.Type==5].OrderID.nunique())

191015
3020
171126
18385
1


# Pivot Table

How can we  identify all limited orders that are ultimately completely executed or completely cancelled. 

**An order is completely executed when the submitted size = the total executed size.**

**An order is completely cancelled when the submitted size = the total deleted size.**

To check these conditions for each order, we have to first create a two-way pivot table to facilitate the analysis.

In [9]:
import numpy as np
table=pd.pivot_table(df_message,
                     index=['OrderID'],  #Row variable of pivot table
                     columns=['Type'],   #Column variable of pivot table
                     values='Size',      #Data to be summarized in pivot table
                     fill_value=0,       #How to replace NaN
                     aggfunc=sum         #How to summarize the data (sum, len, or np.mean) 
                    )   
table[10020:10030]

Type,1,2,3,4,5
OrderID,,,,,
35289538,200,0,200,0,0
35290800,100,0,0,100,0
35293877,100,0,100,0,0
35294546,200,0,200,0,0
35295629,200,0,200,0,0
35295685,200,0,200,0,0
35297076,200,0,200,0,0
35297093,200,0,200,0,0
35309517,100,0,100,0,0


The IDs of the orders that were completely deleted:

In [10]:
ID_deleted = list(table[(table[1]==table[3]) & (table[1]>0)].index)
ID_deleted[0:5]

[9433882, 10296777, 13126986, 16113575, 16113584]

The IDs of the orders that were completely executed:

In [11]:
ID_executed = list(table[(table[1]==table[4]) & (table[1]>0)].index)
ID_deleted[0:5]

[9433882, 10296777, 13126986, 16113575, 16113584]

## Life time of an order

We can also use **groupby** to calculate the life time of each order.

In [12]:
#Order (OrderID 0 removed) and their lifetimes
df_temp=df_message[df_message.OrderID!=0]
df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min()

OrderID
28825        0.000000
29948        0.000000
877345       0.000000
877362       0.000000
877365       0.000000
               ...   
287145896    0.000000
287147169    0.060813
287147170    0.060708
287150868    0.469098
287150931    0.000000
Name: Time, Length: 198667, dtype: float64

In [13]:
#Order (OrderID 0 removed) and their average lifetime
df_temp=df_message[df_message.OrderID!=0]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

3.3616065860485915

For example, if we want to see the average life time of all completely executed orders, we can do

In [14]:
df_temp = df_message[df_message.OrderID.isin(ID_executed)]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

14.604394263057056

Another example, if we want to see the average life time of all deleted orders during 9:30AM to 10:30AM, we can do

In [15]:
df_temp = df_message[(df_message.Time>=34200) & (df_message.Time<=37800) & df_message.OrderID.isin(ID_deleted)]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

1.8622486211823908

## Create bins for each minute

We can split messages and order books into bins (intervals) with each in a length of 1 minute.

In [16]:
import numpy as np
#np.arange(start,stop,stepsize) generate a sequence of spaced numbers
bins = np.arange(34200,57660,60)
df_orderbook['binned'] = pd.cut(df_message.Time, bins)
df_orderbook.head()

,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,binned
0,5859400,200,5853300,18,5859800,200,5853000,150,5861000,200,...,300,5876500,1160,5843800,200,5879000,500,5842700,300,"(34200, 34260]"
1,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,...,300,5876500,1160,5845300,300,5879000,500,5843800,200,"(34200, 34260]"
2,5859400,200,5853300,18,5859800,200,5853200,18,5861000,200,...,300,5876500,1160,5846500,300,5879000,500,5845300,300,"(34200, 34260]"
3,5859100,18,5853300,18,5859400,200,5853200,18,5859800,200,...,300,5873900,100,5846500,300,5876500,1160,5845300,300,"(34200, 34260]"
4,5859100,18,5853300,18,5859200,18,5853200,18,5859400,200,...,300,5871000,10,5846500,300,5873900,100,5845300,300,"(34200, 34260]"


In [17]:
#The last order book in each minute
df_orderbook_minute = df_orderbook.groupby(by="binned").tail(1).copy()
df_orderbook_minute.head()

,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,binned
1264,5856300,205,5853900,18,5856500,980,5853800,2,5857200,100,...,200,5859900,15,5851000,305,5860000,960,5850500,101,"(34200, 34260]"
2862,5853000,100,5848500,100,5853800,100,5846900,100,5854000,350,...,205,5859000,500,5845500,100,5859300,59,5845300,300,"(34260, 34320]"
3636,5856400,980,5853200,200,5857200,200,5853000,75,5858000,300,...,100,5860000,960,5848000,2000,5861000,100,5847900,100,"(34320, 34380]"
6354,5869500,3,5867800,100,5869700,100,5867500,60,5869900,260,...,300,5874100,132,5858600,100,5874300,200,5858500,100,"(34380, 34440]"
7998,5874500,100,5871500,100,5874600,100,5870500,450,5875000,15,...,50,5877900,60,5865000,75,5878000,75,5862500,58,"(34440, 34500]"


Can we use **np.arange** to generate the column indexes for the bid sizes and ask sizes in df_orderbook?

If yes, you will be able to use those column IDs to extract the columns of bid sizes and ask sizes separately. Then you can use .sum(axis=1) to calculate the sum of each row, which is the total number of buy or sell shares. 

For example:

In [18]:
#Create column indexes
import numpy as np
askvolumeid = np.arange(1,40,4)
bidvolumeid = np.arange(3,40,4)
print(askvolumeid)
print(bidvolumeid)

[ 1  5  9 13 17 21 25 29 33 37]
[ 3  7 11 15 19 23 27 31 35 39]


In [19]:
#Get the total bid size and the toal ask size
df_orderbook_minute["totalbid"]=df_orderbook_minute.iloc[:,bidvolumeid].sum(axis=1)
df_orderbook_minute["totalask"]=df_orderbook_minute.iloc[:,askvolumeid].sum(axis=1)
df_orderbook_minute.head()

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,binned,totalbid,totalask
1264,5856300,205,5853900,18,5856500,980,5853800,2,5857200,100,...,15,5851000,305,5860000,960,5850500,101,"(34200, 34260]",1232,2924
2862,5853000,100,5848500,100,5853800,100,5846900,100,5854000,350,...,500,5845500,100,5859300,59,5845300,300,"(34260, 34320]",870,2459
3636,5856400,980,5853200,200,5857200,200,5853000,75,5858000,300,...,960,5848000,2000,5861000,100,5847900,100,"(34320, 34380]",3264,2939
6354,5869500,3,5867800,100,5869700,100,5867500,60,5869900,260,...,132,5858600,100,5874300,200,5858500,100,"(34380, 34440]",1214,2540
7998,5874500,100,5871500,100,5874600,100,5870500,450,5875000,15,...,60,5865000,75,5878000,75,5862500,58,"(34440, 34500]",1258,1328


In [20]:
#Midquote
df_orderbook_minute['Midquote'] = (df_orderbook_minute.iloc[:,0]+df_orderbook_minute.iloc[:,2])/2
df_orderbook_minute.head()

,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,binned,totalbid,totalask,Midquote
1264,5856300,205,5853900,18,5856500,980,5853800,2,5857200,100,...,5851000,305,5860000,960,5850500,101,"(34200, 34260]",1232,2924,5855100.0
2862,5853000,100,5848500,100,5853800,100,5846900,100,5854000,350,...,5845500,100,5859300,59,5845300,300,"(34260, 34320]",870,2459,5850750.0
3636,5856400,980,5853200,200,5857200,200,5853000,75,5858000,300,...,5848000,2000,5861000,100,5847900,100,"(34320, 34380]",3264,2939,5854800.0
6354,5869500,3,5867800,100,5869700,100,5867500,60,5869900,260,...,5858600,100,5874300,200,5858500,100,"(34380, 34440]",1214,2540,5868650.0
7998,5874500,100,5871500,100,5874600,100,5870500,450,5875000,15,...,5865000,75,5878000,75,5862500,58,"(34440, 34500]",1258,1328,5873000.0


## Correlation

In [21]:
#How to compute the correlation between two sequences of numbers
data1=[3,1,2,3,2,4,1]
data2=[-2,-4,-4,-10,-5,-3,-7]
np.corrcoef(data1, data2)

array([[1.        , 0.16593663],
       [0.16593663, 1.        ]])